In [ ]:
```xml
<VSCode.Cell language="markdown">
# DESI Spectra Embeddings Visualization

This notebook visualizes the embeddings extracted from DESI spectra using the trained AstroPT model.

We use dimensionality reduction techniques (UMAP and t-SNE) to project the high-dimensional embeddings into 2D space for visualization.
</VSCode.Cell>
<VSCode.Cell language="python">
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configure matplotlib
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 1. Load Embeddings

First, load the extracted embeddings and associated metadata (target IDs and redshifts).
</VSCode.Cell>
<VSCode.Cell language="python">
# Paths to saved embeddings (update these paths as needed)
embeddings_file = "desi_embeddings.npy"
targetids_file = "desi_targetids.npy"
redshifts_file = "desi_redshifts.npy"

# Load data
print("Loading embeddings...")
embeddings = np.load(embeddings_file)
target_ids = np.load(targetids_file)
redshifts = np.load(redshifts_file)

print(f"Embeddings shape: {embeddings.shape}")
print(f"Number of spectra: {len(target_ids)}")
print(f"Embedding dimension: {embeddings.shape[1]}")
print(f"Redshift range: [{redshifts.min():.3f}, {redshifts.max():.3f}]")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 2. Basic Statistics

Examine the distribution of redshifts and embedding statistics.
</VSCode.Cell>
<VSCode.Cell language="python">
# Plot redshift distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(redshifts, bins=50, alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Redshift', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Redshift Distribution', fontsize=14)
axes[0].grid(alpha=0.3)

# Cumulative distribution
axes[1].hist(redshifts, bins=50, cumulative=True, alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Redshift', fontsize=12)
axes[1].set_ylabel('Cumulative Count', fontsize=12)
axes[1].set_title('Cumulative Redshift Distribution', fontsize=14)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Print statistics
print(f"\nRedshift statistics:")
print(f"  Mean: {redshifts.mean():.3f}")
print(f"  Median: {np.median(redshifts):.3f}")
print(f"  Std: {redshifts.std():.3f}")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 3. UMAP Dimensionality Reduction

Use UMAP (Uniform Manifold Approximation and Projection) to reduce embeddings to 2D.
UMAP is particularly good at preserving both local and global structure.
</VSCode.Cell>
<VSCode.Cell language="python">
import umap
import time

print("Running UMAP dimensionality reduction...")
print("This may take a few minutes...")

# Configure UMAP
umap_model = umap.UMAP(
    n_neighbors=15,      # Size of local neighborhood
    min_dist=0.1,        # Minimum distance between points in low-dim space
    n_components=2,      # Target dimensions
    metric='cosine',     # Distance metric (cosine is good for embeddings)
    random_state=42      # For reproducibility
)

# Fit and transform
start_time = time.time()
embeddings_umap = umap_model.fit_transform(embeddings)
elapsed = time.time() - start_time

print(f"✓ UMAP completed in {elapsed:.2f} seconds")
print(f"UMAP embeddings shape: {embeddings_umap.shape}")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 4. Visualize UMAP Results

Plot the 2D UMAP projection, colored by redshift.
</VSCode.Cell>
<VSCode.Cell language="python">
fig, ax = plt.subplots(figsize=(12, 10))

# Create scatter plot colored by redshift
scatter = ax.scatter(
    embeddings_umap[:, 0],
    embeddings_umap[:, 1],
    c=redshifts,
    cmap='viridis',
    s=5,
    alpha=0.6,
    edgecolors='none'
)

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Redshift', fontsize=12)

ax.set_xlabel('UMAP 1', fontsize=12)
ax.set_ylabel('UMAP 2', fontsize=12)
ax.set_title('DESI Spectra Embeddings (UMAP projection, colored by redshift)', fontsize=14)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('umap_desi_embeddings_redshift.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: umap_desi_embeddings_redshift.png")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 5. Redshift Binning

Visualize different redshift bins separately to see how the embedding space is organized.
</VSCode.Cell>
<VSCode.Cell language="python">
# Define redshift bins
z_bins = [0, 0.5, 1.0, 1.5, 2.0, np.inf]
z_labels = ['z < 0.5', '0.5 ≤ z < 1.0', '1.0 ≤ z < 1.5', '1.5 ≤ z < 2.0', 'z ≥ 2.0']
colors_bins = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

# Assign each spectrum to a bin
z_bin_indices = np.digitize(redshifts, z_bins) - 1

fig, ax = plt.subplots(figsize=(14, 10))

# Plot each bin with different color
for i, label in enumerate(z_labels):
    mask = z_bin_indices == i
    if mask.sum() > 0:
        ax.scatter(
            embeddings_umap[mask, 0],
            embeddings_umap[mask, 1],
            c=colors_bins[i],
            label=f'{label} (n={mask.sum()})',
            s=8,
            alpha=0.5,
            edgecolors='none'
        )

ax.set_xlabel('UMAP 1', fontsize=12)
ax.set_ylabel('UMAP 2', fontsize=12)
ax.set_title('DESI Spectra Embeddings by Redshift Bins', fontsize=14)
ax.legend(loc='best', fontsize=10, markerscale=2)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('umap_desi_embeddings_bins.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: umap_desi_embeddings_bins.png")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 6. t-SNE Dimensionality Reduction (Optional)

t-SNE is another popular dimensionality reduction technique. It's good at preserving local structure
but can be slower than UMAP for large datasets.
</VSCode.Cell>
<VSCode.Cell language="python">
from sklearn.manifold import TSNE

print("Running t-SNE dimensionality reduction...")
print("This may take several minutes for large datasets...")

# Configure t-SNE
tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate=200,
    n_iter=1000,
    random_state=42,
    verbose=1
)

# Fit and transform (may take a while)
start_time = time.time()
embeddings_tsne = tsne.fit_transform(embeddings)
elapsed = time.time() - start_time

print(f"✓ t-SNE completed in {elapsed:.2f} seconds")
print(f"t-SNE embeddings shape: {embeddings_tsne.shape}")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 7. Visualize t-SNE Results
</VSCode.Cell>
<VSCode.Cell language="python">
fig, ax = plt.subplots(figsize=(12, 10))

# Create scatter plot colored by redshift
scatter = ax.scatter(
    embeddings_tsne[:, 0],
    embeddings_tsne[:, 1],
    c=redshifts,
    cmap='viridis',
    s=5,
    alpha=0.6,
    edgecolors='none'
)

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Redshift', fontsize=12)

ax.set_xlabel('t-SNE 1', fontsize=12)
ax.set_ylabel('t-SNE 2', fontsize=12)
ax.set_title('DESI Spectra Embeddings (t-SNE projection, colored by redshift)', fontsize=14)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('tsne_desi_embeddings_redshift.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: tsne_desi_embeddings_redshift.png")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 8. Density Plots

Create density plots to show where embeddings are concentrated.
</VSCode.Cell>
<VSCode.Cell language="python">
from scipy.stats import gaussian_kde

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# UMAP density
xy_umap = np.vstack([embeddings_umap[:, 0], embeddings_umap[:, 1]])
z_umap = gaussian_kde(xy_umap)(xy_umap)

axes[0].scatter(
    embeddings_umap[:, 0],
    embeddings_umap[:, 1],
    c=z_umap,
    s=5,
    cmap='plasma',
    alpha=0.6,
    edgecolors='none'
)
axes[0].set_xlabel('UMAP 1', fontsize=12)
axes[0].set_ylabel('UMAP 2', fontsize=12)
axes[0].set_title('UMAP Embedding Density', fontsize=14)
axes[0].grid(alpha=0.2)

# t-SNE density
xy_tsne = np.vstack([embeddings_tsne[:, 0], embeddings_tsne[:, 1]])
z_tsne = gaussian_kde(xy_tsne)(xy_tsne)

axes[1].scatter(
    embeddings_tsne[:, 0],
    embeddings_tsne[:, 1],
    c=z_tsne,
    s=5,
    cmap='plasma',
    alpha=0.6,
    edgecolors='none'
)
axes[1].set_xlabel('t-SNE 1', fontsize=12)
axes[1].set_ylabel('t-SNE 2', fontsize=12)
axes[1].set_title('t-SNE Embedding Density', fontsize=14)
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.savefig('density_desi_embeddings.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: density_desi_embeddings.png")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 9. Save Reduced Embeddings

Save the 2D projections for later use or analysis.
</VSCode.Cell>
<VSCode.Cell language="python">
# Save UMAP projections
np.save('desi_embeddings_umap2d.npy', embeddings_umap)
print("Saved: desi_embeddings_umap2d.npy")

# Save t-SNE projections
np.save('desi_embeddings_tsne2d.npy', embeddings_tsne)
print("Saved: desi_embeddings_tsne2d.npy")

# Optionally save as a combined file with metadata
output_data = {
    'embeddings_umap': embeddings_umap,
    'embeddings_tsne': embeddings_tsne,
    'target_ids': target_ids,
    'redshifts': redshifts
}
np.savez('desi_embeddings_with_metadata.npz', **output_data)
print("Saved: desi_embeddings_with_metadata.npz")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 10. Analysis Summary

Examine the structure and clustering in the embedding space.
</VSCode.Cell>
<VSCode.Cell language="python">
print("=" * 60)
print("EMBEDDING SPACE ANALYSIS SUMMARY")
print("=" * 60)
print(f"\nTotal spectra: {len(embeddings)}")
print(f"Embedding dimension: {embeddings.shape[1]}")
print(f"\nRedshift statistics:")
print(f"  Range: [{redshifts.min():.3f}, {redshifts.max():.3f}]")
print(f"  Mean ± Std: {redshifts.mean():.3f} ± {redshifts.std():.3f}")
print(f"  Median: {np.median(redshifts):.3f}")

# Calculate variance explained in first few PCs
from sklearn.decomposition import PCA
pca = PCA(n_components=10)
pca.fit(embeddings)
print(f"\nPCA variance explained (first 10 components):")
for i, var in enumerate(pca.explained_variance_ratio_[:10], 1):
    print(f"  PC{i}: {var:.3%}")
print(f"  Cumulative (PC1-10): {pca.explained_variance_ratio_[:10].sum():.3%}")

print("\n" + "=" * 60)
print("✓ Visualization complete!")
print("=" * 60)
</VSCode.Cell>